# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive, step-by-step guide to loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if needed)
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema. This provides programmatic access to the dataset structure and content with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Explore available record sets, fields, and columns. Reference each by its `@id` (the unique identifier in the Croissant schema), not just its name.

In [ ]:
# List available record sets and their field @ids

print("Available record sets (by @id):\n")
all_record_sets = metadata.record_sets
if not all_record_sets:
    print("No record sets found in the top level. Attempting to traverse record sets via files...")
    files = metadata.files
    for file in files:
        if hasattr(file, 'record_sets'):
            for rset in file.record_sets:
                print(f"  @id: {rset.id}, Name: {rset.name or ''}")
else:
    for rset in all_record_sets:
        print(f"  @id: {rset.id}, Name: {rset.name or ''}")

# For this dataset, let's attempt to find the main record set (often there is only one prominent one).
try:
    main_record_sets = metadata.record_sets
    if main_record_sets and len(main_record_sets) > 0:
        main_record_set = main_record_sets[0]
        print(f"\nFields in record set '{main_record_set.id}':")
        for field in main_record_set.fields:
            print(f"   Field @id: {field.id}, Name: {field.name}, dataType: {field.data_type}")
    else:
        # Try files
        files = metadata.files
        found = False
        for file in files:
            if hasattr(file, 'record_sets') and file.record_sets:
                rset = file.record_sets[0]
                print(f"\nFields in record set '{rset.id}':")
                for field in rset.fields:
                    print(f"   Field @id: {field.id}, Name: {field.name}, dataType: {field.data_type}")
                found = True
        if not found:
            print("No fields found via files + record sets.")
except Exception as e:
    print(f"Error traversing record sets: {e}")

## 3. Data Extraction
Load the records from a selected record set into pandas DataFrames for analysis.

To extract records, we reference the record set's `@id`.

In [ ]:
# Discover available record sets
all_record_sets = metadata.record_sets

if not all_record_sets or len(all_record_sets) == 0:
    # Try extracting from files
    record_sets = []
    for file in metadata.files:
        if hasattr(file, 'record_sets'):
            for rset in file.record_sets:
                record_sets.append(rset.id)
else:
    record_sets = [rset.id for rset in all_record_sets]

print(f"Record sets: {record_sets}")
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from record set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns in {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    except Exception as e:
        print(f"  Could not load records: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple analyses with the records from a selected record set.

We'll select a numeric field (referenced by its `@id`) and perform:
- Basic filtering
- Normalization
- Grouping

Please check the columns printed above and replace `<numeric_field_id>` and `<group_field_id>` with appropriate column names as needed.

In [ ]:
# Select the main record set and a numeric field
chosen_record_set_id = record_sets[0] if len(record_sets) > 0 else None
df = dataframes[chosen_record_set_id]
print(f"Available columns in {chosen_record_set_id}:")
print(df.columns.tolist())

# Replace with a real numeric column @id from the above list (e.g., 'coverage', 'unique_peptides', etc.)
numeric_field_id = None
# Example: Find a float/int column
for col in df.columns:
    # Guess column types for numeric columns
    if df[col].dtype in ['float64', 'int64']:
        numeric_field_id = col
        break
if numeric_field_id is None:
    for col in df.columns:
        # Fallback: Try parsing columns
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype in ['float64', 'int64']:
                numeric_field_id = col
                break
        except:
            continue

print(f"Using numeric field: {numeric_field_id}")

# Filtering
threshold = df[numeric_field_id].mean() if numeric_field_id else None

if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a likely categorical field (e.g. 'modification', 'sample' etc.)
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break
    print(f"Using group field: {group_field_id}")

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}, showing mean {numeric_field_id} per group:")
        display(grouped_df.head())
else:
    print("No numeric field found to analyze.")

## 5. Visualization
Visualize distributions or relationships. For example, histogram of a numeric field or bar chart grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
if group_field_id and numeric_field_id:
    plt.figure(figsize=(10,5))
    sns.barplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (filtered)")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded the Croissant FAIR² mass spectrometry dataset and inspected record sets and their fields using their respective `@id`s.
- We extracted the records, identified numeric and categorical fields, and performed filtering, normalization, and grouping.
- We visualized numeric distributions and group means across relevant attributes.

Feel free to further explore the dataset by adjusting the field selections above or applying additional analyses as needed.